# 18. Supervised Learning: Naive Bayes

## Algorithm Category
**Type**: Supervised Learning - Classification  
**Complexity**: Low  
**Use Case**: Probabilistic classification based on Bayes' theorem

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand Bayes' theorem and the naive independence assumption
- Implement different Naive Bayes variants (Gaussian, Multinomial, Bernoulli)
- Understand when to use each variant
- Handle text classification with Multinomial Naive Bayes
- Apply Naive Bayes to real-world classification problems

## Historical Context

Naive Bayes is based on Bayes' theorem from probability theory:
- Thomas Bayes (1701-1761): Developed Bayes' theorem
- The "naive" assumption (feature independence) was formalized in the 1960s
- Widely used in text classification and spam filtering

**Key Papers/References:**
- Bayes, T. (1763). "An Essay towards solving a Problem in the Doctrine of Chances"
- Domingos, P. & Pazzani, M. (1997). "On the optimality of the simple Bayesian classifier"

## When to Use Naive Bayes

Naive Bayes is appropriate when:
- You need a fast, simple classifier
- Working with text data (email spam, document classification)
- Features are conditionally independent (or approximately so)
- Small training datasets
- Real-time predictions needed
- High-dimensional data (works well with many features)

## Theory & Mechanics

### Mathematical Foundation

Naive Bayes uses Bayes' theorem with the "naive" assumption of feature independence.

**Bayes' Theorem:**
$$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$$

**Naive Assumption (Independence):**
$$P(X|y) = P(x_1, x_2, ..., x_n|y) = \prod_{i=1}^{n} P(x_i|y)$$

**Classification Rule:**
$$\hat{y} = \arg\max_{y} P(y) \prod_{i=1}^{n} P(x_i|y)$$

### Variants

1. **Gaussian Naive Bayes**: For continuous features
   - Assumes features follow Gaussian distribution
   - $P(x_i|y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$

2. **Multinomial Naive Bayes**: For discrete counts (e.g., word counts)
   - Uses multinomial distribution
   - $P(x_i|y) = \frac{N_{yi} + \alpha}{N_y + \alpha n}$

3. **Bernoulli Naive Bayes**: For binary features
   - Uses Bernoulli distribution
   - $P(x_i|y) = P(i|y)x_i + (1 - P(i|y))(1 - x_i)$

### How It Works

1. **Training**: Estimate $P(y)$ and $P(x_i|y)$ for each class and feature
2. **Prediction**: Calculate $P(y|X)$ for each class using Bayes' theorem
3. **Decision**: Choose class with highest probability

### Key Hyperparameters

- **alpha (smoothing)**: Additive smoothing parameter (prevents zero probabilities)
- **fit_prior**: Whether to learn class prior probabilities
- **class_prior**: Prior probabilities of classes

### Limitations

- Strong independence assumption (often violated in practice)
- Can be outperformed by more sophisticated methods
- Requires feature independence (or approximate independence)
- May struggle with correlated features


## Implementation

Let's implement different Naive Bayes variants.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, fetch_20newsgroups
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier
from src.models.classification import calculate_classification_metrics, plot_confusion_matrix
from src.utils.benchmarking import benchmark_model_training
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Example 1: Gaussian Naive Bayes (continuous features)
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='Species')

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {iris.target_names.tolist()}")

# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# Train Gaussian Naive Bayes
model = GaussianNB()
model.fit(X_train, y_train)

print("\nGaussian Naive Bayes:")
print(f"Class priors: {model.class_prior_}")
print(f"Number of classes: {len(model.classes_)}")

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.3f}")

# Get probability estimates
y_pred_proba = model.predict_proba(X_test)
print(f"\nSample prediction probabilities:")
print(f"  Sample 0: {y_pred_proba[0]}")
print(f"  Predicted class: {iris.target_names[y_pred[0]]}")


In [ ]:
# Example 2: Multinomial Naive Bayes (for text/count data)
# Using Breast Cancer dataset (convert to counts/discrete)
cancer = load_breast_cancer()
X_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_cancer = pd.Series(cancer.target, name='Target')

# Discretize features (convert to integer counts for demonstration)
X_cancer_discrete = (X_cancer * 10).astype(int)

X_c_train, X_c_test, y_c_train, y_c_test = split_data(X_cancer_discrete, y_cancer, test_size=0.2, random_state=42)

# Train Multinomial Naive Bayes
model_multi = MultinomialNB(alpha=1.0)
model_multi.fit(X_c_train, y_c_train)

y_c_pred = model_multi.predict(X_c_test)
cancer_accuracy = accuracy_score(y_c_test, y_c_pred)

print("Multinomial Naive Bayes (on discretized continuous data):")
print(f"  Test Accuracy: {cancer_accuracy:.3f}")
print(f"  Classes: {cancer.target_names.tolist()}")


## Text Classification Example

Let's use Multinomial Naive Bayes for text classification (its primary use case).


In [ ]:
# Text classification example (simplified - using synthetic text data)
# In practice, you'd use real text datasets like 20newsgroups

# Create simple text classification example
texts = [
    "machine learning is great",
    "python programming language",
    "data science algorithms",
    "deep learning neural networks",
    "buy now cheap price",
    "sale discount offer",
    "limited time deal",
    "special promotion today"
]
labels = [0, 0, 0, 0, 1, 1, 1, 1]  # 0 = tech, 1 = spam

# Convert text to features using CountVectorizer
vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(texts)
y_text = np.array(labels)

print(f"Text features shape: {X_text.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Sample words: {list(vectorizer.vocabulary_.keys())[:10]}")

# Train Multinomial Naive Bayes
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text, y_text, test_size=0.25, random_state=42
)

model_text = MultinomialNB(alpha=1.0)
model_text.fit(X_text_train, y_text_train)

y_text_pred = model_text.predict(X_text_test)
text_accuracy = accuracy_score(y_text_test, y_text_pred)

print(f"\nText Classification Results:")
print(f"  Test Accuracy: {text_accuracy:.3f}")
print(f"  Predictions: {y_text_pred}")
print(f"  Actual: {y_text_test}")


## Validation & Testing

Let's validate our models and compare different variants.


In [ ]:
# Validation 1: Compare different Naive Bayes variants
variants = {
    'Gaussian': GaussianNB(),
    'Multinomial': MultinomialNB(alpha=1.0),
    'Bernoulli': BernoulliNB(alpha=1.0)
}

# Use Iris dataset for comparison
results = {}
for name, nb_model in variants.items():
    if name == 'Multinomial':
        # Discretize for Multinomial
        X_discrete = (X * 10).astype(int)
        nb_model.fit(X_discrete.iloc[X_train.index], y_train)
        pred = nb_model.predict(X_discrete.iloc[X_test.index])
    elif name == 'Bernoulli':
        # Binarize for Bernoulli
        X_binary = (X > X.median()).astype(int)
        nb_model.fit(X_binary.iloc[X_train.index], y_train)
        pred = nb_model.predict(X_binary.iloc[X_test.index])
    else:
        nb_model.fit(X_train, y_train)
        pred = nb_model.predict(X_test)
    
    acc = accuracy_score(y_test, pred)
    results[name] = acc
    print(f"{name} Naive Bayes: Accuracy = {acc:.3f}")

best_variant = max(results, key=results.get)
print(f"\nBest variant: {best_variant}")


In [ ]:
# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")

# Validation 3: Check model output
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
print("\n✓ Validation checks passed")


## Real-World Application

Let's tune hyperparameters and visualize class probabilities.


In [ ]:
# Hyperparameter tuning (smoothing parameter)
alphas = [0.1, 0.5, 1.0, 2.0, 5.0]
alpha_scores = []

for alpha in alphas:
    nb = MultinomialNB(alpha=alpha)
    # Use discretized data
    X_discrete = (X * 10).astype(int)
    scores = cross_val_score(nb, X_discrete, y, cv=5, scoring='accuracy')
    alpha_scores.append(scores.mean())
    print(f"Alpha={alpha}: CV Accuracy = {scores.mean():.3f}")

optimal_alpha = alphas[np.argmax(alpha_scores)]
print(f"\nOptimal alpha: {optimal_alpha}")

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(alphas, alpha_scores, 'o-')
plt.axvline(x=optimal_alpha, color='r', linestyle='--', label=f'Optimal α={optimal_alpha}')
plt.xlabel('Smoothing Parameter (α)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Naive Bayes: Effect of Smoothing Parameter')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize class probabilities
proba_df = pd.DataFrame(y_pred_proba, columns=iris.target_names)
proba_df['predicted'] = [iris.target_names[p] for p in y_pred]
proba_df['actual'] = [iris.target_names[a] for a in y_test.values]

print("Sample Predictions with Probabilities:")
print(proba_df.head(10))

# Plot probability distributions
plt.figure(figsize=(12, 5))
for idx, class_name in enumerate(iris.target_names):
    plt.subplot(1, 3, idx + 1)
    plt.hist(y_pred_proba[y_test.values == idx, idx], bins=20, alpha=0.7, edgecolor='black')
    plt.xlabel('Predicted Probability')
    plt.ylabel('Frequency')
    plt.title(f'Probability Distribution: {class_name}')
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **Naive Bayes Basics**
   - Based on Bayes' theorem with independence assumption
   - Fast training and prediction
   - Provides probability estimates

2. **Variants**
   - **Gaussian**: For continuous features (assumes normal distribution)
   - **Multinomial**: For count data (text classification, word counts)
   - **Bernoulli**: For binary features

3. **Key Parameters**
   - **alpha (smoothing)**: Prevents zero probabilities (Laplace smoothing)
   - **fit_prior**: Whether to learn class priors from data

4. **Best Practices**
   - Use appropriate variant for your data type
   - Tune smoothing parameter (alpha)
   - Works well for text classification
   - Fast baseline for comparison

### When to Use Naive Bayes

✅ **Good for:**
- Text classification (spam detection, sentiment analysis)
- High-dimensional data
- Small training datasets
- Real-time predictions
- When you need probability estimates
- Fast baseline classifier

❌ **Not ideal for:**
- Correlated features (independence assumption violated)
- Complex relationships between features
- When highest accuracy is required
- Very small datasets (may overfit)

### Next Steps

- Try **Complement Naive Bayes** for imbalanced text classification
- Explore **Bayesian Networks** for handling dependencies
- Compare with **Logistic Regression** for similar use cases
- Use for **spam filtering** and **document classification**
